# Image Flow Matching: From Theory to Generation

This notebook walks through **continuous flow matching** applied to image generation on MNIST. We cover three progressively advanced topics:

1. **Unconditional Flow Matching** — train a U-Net to learn the velocity field from noise to images
2. **Class-Conditional Generation with Classifier-Free Guidance (CFG)** — control *what* digit to generate
3. **Rectified Flow (Reflow)** — straighten trajectories for faster sampling

**Prerequisites:** Familiarity with the Gaussian flow matching walkthrough (see `synthetic_gaussian/`). We build directly on the Conditional Flow Matching (CFM) and Optimal Transport (OT) path concepts introduced there.

**Compute:** Everything runs on a single Colab GPU (T4 or better) in ~15–20 minutes total.

## Setup

In [ ]:
import sys
import os
import math
import copy

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import torchvision
import torchvision.transforms as T
import matplotlib.pyplot as plt
from tqdm import tqdm

sys.path.insert(0, os.path.abspath('.'))

from image_model import (
    UNet, sincos_embed, TimeEmbedding, ResBlock,
    Downsample, Upsample,
)
from image_data import get_mnist_loaders, show_samples, IMG_SIZE, IMG_CHANNELS, NUM_CLASSES

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

---
# Part 1: Unconditional Flow Matching on MNIST

## 1.1 Data

MNIST consists of 60K training images of handwritten digits (28×28, grayscale). Each pixel is normalized to $[0, 1]$.

In [ ]:
BATCH_SIZE = 128
train_loader, test_loader = get_mnist_loaders(batch_size=BATCH_SIZE)

# Visualize some training images
examples, labels = next(iter(train_loader))
show_samples(examples[:32], nrow=8, title="MNIST training samples")
print(f"Image shape: {examples.shape}")
print(f"Pixel range: [{examples.min():.1f}, {examples.max():.1f}]")

\
## 1.2 Review: OT Conditional Flow Matching

Recall from the Gaussian walkthrough: we want to learn a velocity field $v_\theta(x_t, t)$ that transports samples from a base distribution $p_0 = \mathcal{N}(0, I)$ to the data distribution $p_1 = p_\text{data}$.

Using the **Optimal Transport conditional path**, the interpolation between noise $x_0$ and data $x_1$ is:

$$x_t = (1 - t)\, x_0 + t\, x_1, \qquad t \in [0, 1]$$

The target velocity that the model should learn is:

$$u_t(x \mid x_1) = x_1 - x_0$$

And the **CFM loss** is simply the MSE between the model's prediction and this target:

$$\mathcal{L}_\text{CFM}(\theta) = \mathbb{E}_{t \sim \mathcal{U}(0,1),\; x_0 \sim \mathcal{N}(0,I),\; x_1 \sim p_\text{data}} \left\| v_\theta(x_t, t) - (x_1 - x_0) \right\|^2$$

This is the same math as the Gaussian walkthrough — the only difference is that $x_1$ is now a 28×28 image instead of a 1024-dimensional vector, and $v_\theta$ is a **U-Net** instead of a residual MLP.

## 1.3 Visualize the OT Interpolation

Before training, let's see what the interpolation $x_t = (1-t) x_0 + t x_1$ looks like. At $t=0$ we have pure noise, and at $t=1$ we have the clean image.

In [ ]:
# Pick a few real images and show the interpolation path
x1 = examples[:4].to(device)   # real images
x0 = torch.randn_like(x1)      # noise

timesteps = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
fig, axes = plt.subplots(4, len(timesteps), figsize=(12, 7))
for row in range(4):
    for col, t_val in enumerate(timesteps):
        x_t = (1 - t_val) * x0[row] + t_val * x1[row]
        axes[row, col].imshow(x_t[0].cpu().numpy(), cmap='gray', vmin=-2, vmax=2)
        axes[row, col].axis('off')
        if row == 0:
            axes[row, col].set_title(f't = {t_val:.1f}', fontsize=11)
plt.suptitle('OT Interpolation: noise (t=0) → data (t=1)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

\
## 1.4 U-Net Architecture

For images, the velocity field $v_\theta$ is a **U-Net**: a convolutional encoder-decoder with skip connections. This is the standard architecture for diffusion/flow models on images.

Key components:

- **Time embedding:** Scalar $t \in [0,1]$ is encoded via sinusoidal features (same as in the Gaussian walkthrough), then projected through an MLP to produce an embedding vector $e_t$.

- **FiLM conditioning:** Each residual block uses $e_t$ to modulate its features via scale and shift (Feature-wise Linear Modulation):
$$h \leftarrow \text{GroupNorm}(h) \cdot (1 + \gamma) + \beta, \qquad [\gamma, \beta] = \text{Linear}(e_t)$$
This is analogous to the AdaLN mechanism from the MDLM walkthrough, but applied to convolutional features.

- **Encoder-decoder with skip connections:** The encoder downsamples spatially while increasing channels (28×28×64 → 14×14×64 → 14×14×128). The decoder reverses this, concatenating encoder features at matching resolutions via skip connections.

Our U-Net has ~3.5M parameters — small enough to train in minutes on a T4.

In [ ]:
model = UNet(
    in_channels=IMG_CHANNELS,   # 1 (grayscale)
    out_channels=IMG_CHANNELS,  # 1 (predict velocity)
    base_channels=64,
    channel_mults=(1, 2),       # 2 resolution levels: 28→14
    num_res_blocks=2,
    embed_dim=128,
    num_classes=0,              # unconditional for now
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,}")

# Quick shape check
dummy_x = torch.randn(2, 1, 28, 28, device=device)
dummy_t = torch.rand(2, device=device)
out = model(dummy_x, dummy_t)
print(f"Input shape:  {dummy_x.shape}")
print(f"Output shape: {out.shape}")
assert out.shape == dummy_x.shape, "Output shape mismatch!"
print("Shape check passed.")

\
## 1.5 CFM Loss Function

The loss is identical to the Gaussian walkthrough, adapted for image tensors (4D instead of 2D):

1. Sample $x_0 \sim \mathcal{N}(0, I)$ (noise image)
2. Sample $t \sim \mathcal{U}(0, 1)$ (random timestep per sample)
3. Interpolate: $x_t = (1-t) x_0 + t x_1$
4. Predict: $\hat{v} = v_\theta(x_t, t)$
5. Loss: $\|\hat{v} - (x_1 - x_0)\|^2$

In [ ]:
def cfm_loss(model, x1):
    """
    Compute the OT conditional flow matching loss.

    Args:
        model: UNet velocity field
        x1: (B, 1, 28, 28) batch of real images in [0, 1]
    Returns:
        scalar loss
    """
    B = x1.shape[0]

    # sample noise and random timesteps
    x0 = torch.randn_like(x1)
    t = torch.rand(B, device=x1.device)

    # ── interpolate: x_t = (1 - t) * x_0 + t * x_1 ──
    t_img = t[:, None, None, None]          # (B, 1, 1, 1) for broadcasting
    x_t = (1 - t_img) * x0 + t_img * x1

    # ── target velocity: u_t = x_1 - x_0 ──
    target = x1 - x0

    # ── predict and compute MSE loss ──
    pred = model(x_t, t)
    loss = ((pred - target) ** 2).mean()
    return loss

In [ ]:
# Sanity check: loss on a random batch should be around 2.0
# (since both pred and target are roughly unit-Gaussian at initialization)
with torch.no_grad():
    test_loss = cfm_loss(model, examples[:16].to(device))
print(f"Initial loss: {test_loss.item():.4f} (expected ~2.0)")

## 1.6 Training

We train for 10 epochs (~4700 steps). On a T4, this takes about 5–8 minutes. The loss should drop from ~2.0 to ~0.3–0.4.

In [ ]:
N_EPOCHS = 10
LR = 3e-4

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
losses = []

model.train()
for epoch in range(N_EPOCHS):
    epoch_loss = 0.0
    n_batches = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{N_EPOCHS}")
    for images, _ in pbar:
        images = images.to(device)
        loss = cfm_loss(model, images)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        losses.append(loss.item())
        epoch_loss += loss.item()
        n_batches += 1
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    print(f"  Epoch {epoch+1} avg loss: {epoch_loss / n_batches:.4f}")

plt.plot(losses, alpha=0.3, color='blue')
# smoothed
window = 50
if len(losses) > window:
    smoothed = [sum(losses[i:i+window])/window for i in range(len(losses)-window)]
    plt.plot(range(window//2, len(smoothed)+window//2), smoothed, color='blue', linewidth=2)
plt.xlabel("Gradient step")
plt.ylabel("CFM loss")
plt.title("Training Curve (Unconditional)")
plt.grid(alpha=0.3)
plt.show()

\
## 1.7 Sampling via Euler Integration

To generate images, we integrate the learned velocity field forward from $t=0$ to $t=1$ using Euler's method:

$$x_{t + \Delta t} = x_t + v_\theta(x_t, t) \cdot \Delta t, \qquad x_0 \sim \mathcal{N}(0, I)$$

After $T$ steps (with $\Delta t = 1/T$), $x_1$ is our generated image.

In [ ]:
@torch.no_grad()
def euler_sample(model, n_samples, n_steps=50, device='cpu'):
    """
    Generate images by Euler integration of the learned velocity field.

    Args:
        model: trained UNet
        n_samples: number of images to generate
        n_steps: number of Euler steps (more steps = better quality)
    Returns:
        (n_samples, 1, 28, 28) generated images
    """
    model.eval()
    # start from noise
    x = torch.randn(n_samples, 1, 28, 28, device=device)
    dt = 1.0 / n_steps

    for i in range(n_steps):
        t = torch.full((n_samples,), i / n_steps, device=device)
        v = model(x, t)
        x = x + v * dt

    return x

In [ ]:
# Generate and visualize samples
samples = euler_sample(model, n_samples=32, n_steps=50, device=device)
show_samples(samples, nrow=8, title="Generated MNIST digits (unconditional, 50 steps)")

## 1.8 Visualize the Generation Trajectory

We can snapshot the state at intermediate timesteps to see how noise gradually becomes a digit.

In [ ]:
model.eval()
n_show = 4
n_steps = 50
dt = 1.0 / n_steps
snap_times = [0.0, 0.1, 0.2, 0.4, 0.6, 0.8, 1.0]

x = torch.randn(n_show, 1, 28, 28, device=device)
snapshots = {0.0: x.clone()}

with torch.no_grad():
    for i in range(n_steps):
        t_val = i / n_steps
        t = torch.full((n_show,), t_val, device=device)
        x = x + model(x, t) * dt
        # check if we should snapshot
        next_t = (i + 1) / n_steps
        for s in snap_times:
            if t_val < s <= next_t:
                snapshots[s] = x.clone()

fig, axes = plt.subplots(n_show, len(snap_times), figsize=(14, 2.5 * n_show))
for row in range(n_show):
    for col, t_val in enumerate(snap_times):
        img = snapshots[t_val][row, 0].cpu().numpy()
        axes[row, col].imshow(img, cmap='gray')
        axes[row, col].axis('off')
        if row == 0:
            axes[row, col].set_title(f't = {t_val:.1f}', fontsize=11)
plt.suptitle('Generation trajectory: noise → digit', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

\
## 1.9 Effect of Number of Function Evaluations (NFE)

More Euler steps = better ODE approximation = higher quality samples, but slower. Let's compare.

In [ ]:
step_counts = [1, 2, 5, 10, 25, 50, 100]

# Use the same initial noise for fair comparison
torch.manual_seed(42)
z = torch.randn(8, 1, 28, 28, device=device)

fig, axes = plt.subplots(len(step_counts), 8, figsize=(12, 1.8 * len(step_counts)))
model.eval()

for row, n_steps in enumerate(step_counts):
    x = z.clone()
    dt = 1.0 / n_steps
    with torch.no_grad():
        for i in range(n_steps):
            t = torch.full((8,), i / n_steps, device=device)
            x = x + model(x, t) * dt
    for col in range(8):
        axes[row, col].imshow(x[col, 0].cpu().numpy(), cmap='gray')
        axes[row, col].axis('off')
    axes[row, 0].set_ylabel(f'{n_steps} steps', fontsize=10, rotation=0, labelpad=50)

plt.suptitle('Sample quality vs. number of Euler steps (NFE)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

---
# Part 2: Class-Conditional Generation with Classifier-Free Guidance

In Part 1 the model generates random digits — we have no control over *which* digit. Here we add **class conditioning** so we can specify the digit, and use **Classifier-Free Guidance (CFG)** to sharpen the results.

## 2.1 How CFG Works

**Training:**
- Add a learnable class embedding $e_c$ to the time embedding: $e = e_t + e_c$
- With probability $p_\text{uncond}$ (e.g., 10%), replace the real class label with a special **null class** token. This trains the model to also work unconditionally.

**Sampling:**
- Run two forward passes per step: one **conditional** ($v_\text{cond}$) and one **unconditional** ($v_\text{uncond}$, using the null class)
- Blend them with guidance scale $w$:

$$v_\text{guided} = v_\text{uncond} + w \cdot (v_\text{cond} - v_\text{uncond})$$

- $w = 0$: unconditional generation
- $w = 1$: standard conditional generation (no guidance)
- $w > 1$: amplified conditioning — sharper, more class-typical images

## 2.2 Conditional U-Net

The same U-Net architecture, but now with `num_classes=10`. This adds a learnable embedding table mapping class labels {0,...,9} (plus a null class = 10) to embedding vectors that get added to the time embedding.

In [ ]:
cond_model = UNet(
    in_channels=IMG_CHANNELS,
    out_channels=IMG_CHANNELS,
    base_channels=64,
    channel_mults=(1, 2),
    num_res_blocks=2,
    embed_dim=128,
    num_classes=NUM_CLASSES,  # 10 classes + 1 null class
).to(device)

n_params = sum(p.numel() for p in cond_model.parameters())
print(f"Conditional model parameters: {n_params:,}")

## 2.3 Conditional CFM Loss with Class Dropout

In [ ]:
def cfm_loss_conditional(model, x1, class_label, p_uncond=0.1):
    """
    CFM loss with class conditioning and random label dropout for CFG.

    Args:
        model: UNet with num_classes > 0
        x1: (B, 1, 28, 28) real images
        class_label: (B,) integer digit labels 0-9
        p_uncond: probability of replacing label with null class
    """
    B = x1.shape[0]
    device = x1.device

    x0 = torch.randn_like(x1)
    t = torch.rand(B, device=device)

    # OT interpolation
    t_img = t[:, None, None, None]
    x_t = (1 - t_img) * x0 + t_img * x1
    target = x1 - x0

    # ── randomly drop class labels (replace with null class) ──
    class_label = class_label.clone()
    drop_mask = torch.rand(B, device=device) < p_uncond
    class_label[drop_mask] = model.num_classes  # null class = 10

    pred = model(x_t, t, class_label)
    return ((pred - target) ** 2).mean()

## 2.4 Training the Conditional Model

Same training loop, but now we pass class labels to the loss function.

In [ ]:
N_EPOCHS = 10
LR = 3e-4
P_UNCOND = 0.1  # 10% label dropout for CFG

optimizer = torch.optim.Adam(cond_model.parameters(), lr=LR)
cond_losses = []

cond_model.train()
for epoch in range(N_EPOCHS):
    epoch_loss = 0.0
    n_batches = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{N_EPOCHS}")
    for images, labels in pbar:
        images = images.to(device)
        labels = labels.to(device)
        loss = cfm_loss_conditional(cond_model, images, labels, p_uncond=P_UNCOND)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        cond_losses.append(loss.item())
        epoch_loss += loss.item()
        n_batches += 1
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    print(f"  Epoch {epoch+1} avg loss: {epoch_loss / n_batches:.4f}")

plt.plot(cond_losses, alpha=0.3, color='green')
window = 50
if len(cond_losses) > window:
    smoothed = [sum(cond_losses[i:i+window])/window for i in range(len(cond_losses)-window)]
    plt.plot(range(window//2, len(smoothed)+window//2), smoothed, color='green', linewidth=2)
plt.xlabel("Gradient step")
plt.ylabel("CFM loss")
plt.title("Training Curve (Class-Conditional)")
plt.grid(alpha=0.3)
plt.show()

\
## 2.5 Sampling with Classifier-Free Guidance

At each Euler step, we compute two velocities and blend them:

$$v_\text{guided} = v_\text{uncond} + w \cdot (v_\text{cond} - v_\text{uncond})$$

In [ ]:
@torch.no_grad()
def cfg_sample(model, n_samples, class_labels, guidance_scale=2.0,
               n_steps=50, device='cpu'):
    """
    Sample with classifier-free guidance.

    Args:
        model: trained conditional UNet
        n_samples: number of images
        class_labels: (n_samples,) target class for each sample
        guidance_scale: w in the CFG formula (0 = uncond, 1 = cond, >1 = guided)
        n_steps: Euler steps
    Returns:
        (n_samples, 1, 28, 28) generated images
    """
    model.eval()
    x = torch.randn(n_samples, 1, 28, 28, device=device)
    dt = 1.0 / n_steps
    null_labels = torch.full_like(class_labels, model.num_classes)

    for i in range(n_steps):
        t = torch.full((n_samples,), i / n_steps, device=device)

        # conditional and unconditional predictions
        v_cond = model(x, t, class_labels)
        v_uncond = model(x, t, null_labels)

        # CFG blending
        v = v_uncond + guidance_scale * (v_cond - v_uncond)
        x = x + v * dt

    return x

## 2.6 Generate Specific Digits

Let's generate each digit 0–9 with CFG.

In [ ]:
# Generate 4 samples of each digit 0-9
n_per_class = 4
labels = torch.arange(10, device=device).repeat_interleave(n_per_class)
samples = cfg_sample(cond_model, len(labels), labels,
                     guidance_scale=2.0, n_steps=50, device=device)
show_samples(samples, nrow=n_per_class, title="Class-conditional samples (w=2.0)")

## 2.7 Effect of Guidance Scale

Let's see how the guidance scale $w$ affects generation quality for a single digit.

In [ ]:
guidance_scales = [0.0, 0.5, 1.0, 2.0, 4.0, 8.0]
target_digit = 7
n_show = 8

torch.manual_seed(123)
z = torch.randn(n_show, 1, 28, 28, device=device)

fig, axes = plt.subplots(len(guidance_scales), n_show,
                         figsize=(12, 1.8 * len(guidance_scales)))
cond_model.eval()

for row, w in enumerate(guidance_scales):
    x = z.clone()
    dt = 1.0 / 50
    labels = torch.full((n_show,), target_digit, device=device, dtype=torch.long)
    null_labels = torch.full((n_show,), cond_model.num_classes, device=device, dtype=torch.long)

    with torch.no_grad():
        for i in range(50):
            t = torch.full((n_show,), i / 50, device=device)
            v_c = cond_model(x, t, labels)
            v_u = cond_model(x, t, null_labels)
            v = v_u + w * (v_c - v_u)
            x = x + v * dt

    for col in range(n_show):
        axes[row, col].imshow(x[col, 0].cpu().numpy(), cmap='gray')
        axes[row, col].axis('off')
    axes[row, 0].set_ylabel(f'w={w}', fontsize=10, rotation=0, labelpad=35)

plt.suptitle(f'Guidance scale sweep for digit "{target_digit}"', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

\
---
# Part 3: Rectified Flow (Reflow)

## 3.1 The Problem: Curved Trajectories

The OT-CFM model we trained in Part 1 learns curved trajectories — the velocity field changes direction as $t$ goes from 0 to 1. This means we need many Euler steps for accurate integration.

**Idea (Liu et al., 2023):** If the trajectories were **perfectly straight**, we could sample in just **one step**: $x_1 = x_0 + v_\theta(x_0, 0)$. Reflow makes trajectories straighter.

## 3.2 How Reflow Works

1. **Train** a standard flow matching model (done — this is our Part 1 model)
2. **Generate coupling pairs:** For each $x_0 \sim \mathcal{N}(0, I)$, run the ODE forward to get $x_1 = \text{ODE}(x_0)$. Now we have matched $(x_0, x_1)$ pairs where each noise connects to a *specific* image.
3. **Retrain** a new model on these coupled pairs. Because the pairs are already matched (not random), the optimal velocity field is straighter.

The key insight: in standard CFM, each noise sample $x_0$ could be paired with *any* data sample $x_1$. After reflow, each $x_0$ is paired with the *specific* $x_1$ it maps to, so the model learns straighter paths.

\
## 3.3 Measuring Trajectory Straightness

We define straightness as:

$$S = \frac{\|x_1 - x_0\|}{\sum_{i} \|x_{t_{i+1}} - x_{t_i}\|}$$

- $S = 1$: perfectly straight (displacement = path length)
- $S < 1$: curved (path length > displacement)

In [ ]:
def measure_straightness(model, n_samples=64, n_steps=50, device='cpu'):
    """
    Compute average trajectory straightness for the model.

    Straightness = ||x_1 - x_0|| / sum_i ||x_{t+dt} - x_t||
    """
    model.eval()
    x = torch.randn(n_samples, 1, 28, 28, device=device)
    x_start = x.clone()
    dt = 1.0 / n_steps
    path_length = torch.zeros(n_samples, device=device)

    with torch.no_grad():
        for i in range(n_steps):
            t = torch.full((n_samples,), i / n_steps, device=device)
            v = model(x, t)
            step = v * dt
            # accumulate path length: sum of step norms
            path_length += step.flatten(1).norm(dim=1)
            x = x + step

    # displacement: straight-line distance from start to end
    displacement = (x - x_start).flatten(1).norm(dim=1)
    straightness = (displacement / path_length.clamp(min=1e-8)).mean().item()
    return straightness

In [ ]:
orig_straightness = measure_straightness(model, n_samples=128, n_steps=50, device=device)
print(f"Original model trajectory straightness: {orig_straightness:.4f}")
print("(1.0 = perfectly straight, lower = more curved)")

## 3.4 Generate Reflow Pairs

We run the trained model forward to create matched (noise, image) pairs.

In [ ]:
@torch.no_grad()
def generate_reflow_pairs(model, n_pairs, n_steps=50, device='cpu', batch_size=256):
    """
    Generate (x0, x1) pairs by running the trained model forward.

    For each x0 ~ N(0, I), integrate the ODE to get x1 = ODE(x0).
    Returns (noise_samples, generated_images).
    """
    model.eval()
    all_x0 = []
    all_x1 = []

    for start in range(0, n_pairs, batch_size):
        B = min(batch_size, n_pairs - start)
        x0 = torch.randn(B, 1, 28, 28, device=device)
        x = x0.clone()
        dt = 1.0 / n_steps

        # integrate forward
        for i in range(n_steps):
            t = torch.full((B,), i / n_steps, device=device)
            x = x + model(x, t) * dt

        all_x0.append(x0.cpu())
        all_x1.append(x.cpu())

    return torch.cat(all_x0), torch.cat(all_x1)

In [ ]:
print("Generating reflow pairs (this takes ~1 minute)...")
reflow_x0, reflow_x1 = generate_reflow_pairs(
    model, n_pairs=20000, n_steps=50, device=device, batch_size=512
)
print(f"Generated {len(reflow_x0)} pairs")
print(f"Noise shape:  {reflow_x0.shape}")
print(f"Image shape:  {reflow_x1.shape}")

# Show some pairs
fig, axes = plt.subplots(2, 8, figsize=(14, 3.5))
for i in range(8):
    axes[0, i].imshow(reflow_x0[i, 0].numpy(), cmap='gray')
    axes[0, i].axis('off')
    axes[1, i].imshow(reflow_x1[i, 0].clamp(0, 1).numpy(), cmap='gray')
    axes[1, i].axis('off')
axes[0, 0].set_ylabel('x0 (noise)', fontsize=10, rotation=0, labelpad=55)
axes[1, 0].set_ylabel('x1 (image)', fontsize=10, rotation=0, labelpad=55)
plt.suptitle('Reflow coupling pairs', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 3.5 Train the Reflow Model

We train a fresh U-Net on the coupled pairs. The loss is the same CFM loss, but now each noise $x_0$ is paired with a *specific* $x_1$ (not a random data sample).

In [ ]:
# Create a DataLoader from the reflow pairs
reflow_dataset = TensorDataset(reflow_x0, reflow_x1)
reflow_loader = DataLoader(reflow_dataset, batch_size=128, shuffle=True,
                           num_workers=2, pin_memory=True, drop_last=True)

# Fresh model for reflow
reflow_model = UNet(
    in_channels=1, out_channels=1,
    base_channels=64, channel_mults=(1, 2),
    num_res_blocks=2, embed_dim=128, num_classes=0,
).to(device)

print(f"Reflow model parameters: {sum(p.numel() for p in reflow_model.parameters()):,}")

In [ ]:
def cfm_loss_reflow(model, x0, x1):
    """
    CFM loss for reflow: same as standard CFM, but x0 and x1 are
    pre-paired (not independently sampled).

    Args:
        model: UNet velocity field
        x0: (B, 1, 28, 28) noise samples
        x1: (B, 1, 28, 28) corresponding generated images
    """
    B = x0.shape[0]
    t = torch.rand(B, device=x0.device)
    t_img = t[:, None, None, None]

    # same OT interpolation, but with matched pairs
    x_t = (1 - t_img) * x0 + t_img * x1
    target = x1 - x0

    pred = model(x_t, t)
    return ((pred - target) ** 2).mean()

In [ ]:
N_EPOCHS_REFLOW = 10
LR = 3e-4

optimizer = torch.optim.Adam(reflow_model.parameters(), lr=LR)
reflow_losses = []

reflow_model.train()
for epoch in range(N_EPOCHS_REFLOW):
    epoch_loss = 0.0
    n_batches = 0
    pbar = tqdm(reflow_loader, desc=f"Reflow Epoch {epoch+1}/{N_EPOCHS_REFLOW}")
    for x0_batch, x1_batch in pbar:
        x0_batch = x0_batch.to(device)
        x1_batch = x1_batch.to(device)
        loss = cfm_loss_reflow(reflow_model, x0_batch, x1_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        reflow_losses.append(loss.item())
        epoch_loss += loss.item()
        n_batches += 1
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    print(f"  Epoch {epoch+1} avg loss: {epoch_loss / n_batches:.4f}")

plt.plot(reflow_losses, alpha=0.3, color='red')
window = 50
if len(reflow_losses) > window:
    smoothed = [sum(reflow_losses[i:i+window])/window for i in range(len(reflow_losses)-window)]
    plt.plot(range(window//2, len(smoothed)+window//2), smoothed, color='red', linewidth=2)
plt.xlabel("Gradient step")
plt.ylabel("Reflow loss")
plt.title("Training Curve (Reflow)")
plt.grid(alpha=0.3)
plt.show()

## 3.6 Compare Trajectory Straightness

The reflow model should have straighter trajectories than the original.

In [ ]:
reflow_straightness = measure_straightness(reflow_model, n_samples=128, n_steps=50, device=device)
print(f"Original model straightness: {orig_straightness:.4f}")
print(f"Reflow model straightness:   {reflow_straightness:.4f}")
print(f"Improvement: {reflow_straightness - orig_straightness:+.4f}")

## 3.7 Few-Step Sampling: Original vs. Reflow

The payoff: the reflow model should produce better samples with fewer steps.

In [ ]:
step_counts = [1, 2, 5, 10, 25, 50]
n_show = 8

# Same initial noise for both models
torch.manual_seed(42)
z = torch.randn(n_show, 1, 28, 28, device=device)

fig, axes = plt.subplots(len(step_counts), 2 * n_show + 1,
                         figsize=(2 * n_show * 1.2 + 1, 1.5 * len(step_counts)))
# hide separator column
for row in range(len(step_counts)):
    axes[row, n_show].axis('off')

for row, n_steps in enumerate(step_counts):
    dt = 1.0 / n_steps

    # Original model
    x_orig = z.clone()
    with torch.no_grad():
        for i in range(n_steps):
            t = torch.full((n_show,), i / n_steps, device=device)
            x_orig = x_orig + model(x_orig, t) * dt

    # Reflow model
    x_reflow = z.clone()
    with torch.no_grad():
        for i in range(n_steps):
            t = torch.full((n_show,), i / n_steps, device=device)
            x_reflow = x_reflow + reflow_model(x_reflow, t) * dt

    for col in range(n_show):
        axes[row, col].imshow(x_orig[col, 0].cpu().numpy(), cmap='gray')
        axes[row, col].axis('off')
        axes[row, n_show + 1 + col].imshow(x_reflow[col, 0].cpu().numpy(), cmap='gray')
        axes[row, n_show + 1 + col].axis('off')

    axes[row, 0].set_ylabel(f'{n_steps}', fontsize=10, rotation=0, labelpad=25)

# Column headers
axes[0, n_show // 2].set_title('Original', fontsize=12, fontweight='bold')
axes[0, n_show + 1 + n_show // 2].set_title('Reflow', fontsize=12, fontweight='bold')
fig.text(0.02, 0.5, 'Steps', fontsize=12, rotation=90, va='center')
plt.suptitle('Few-step sampling: Original vs Reflow', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
# Part 4: Advanced Sampling Techniques

These exercises use the unconditional MNIST model trained in Part 1. We explore better ODE solvers, noise-space interpolation, and flow-based image editing.

\
## 4.1 Heun's Method (2nd-Order ODE Solver)

Euler's method is first-order: it uses the velocity at the *start* of each step. **Heun's method** (also called the improved Euler or explicit trapezoidal method) is second-order: it averages the velocity at the start and end of each step.

Given the ODE $\frac{dx}{dt} = v_\theta(x, t)$:

**Euler** (1 model evaluation per step):
$$x_{t+\Delta t} = x_t + \Delta t \cdot v_\theta(x_t, t)$$

**Heun** (2 model evaluations per step):
$$\tilde{x} = x_t + \Delta t \cdot v_\theta(x_t, t) \qquad \text{(Euler predictor)}$$
$$x_{t+\Delta t} = x_t + \frac{\Delta t}{2}\bigl[v_\theta(x_t, t) + v_\theta(\tilde{x}, t + \Delta t)\bigr] \qquad \text{(trapezoidal corrector)}$$

Heun uses **2 NFE per step** but gives much better accuracy per step — so at the same total NFE budget, it often beats Euler.

In [ ]:
@torch.no_grad()
def heun_sample(model, n_samples, n_steps=25, device='cpu'):
    """
    Generate images using Heun's method (2nd-order ODE solver).
    Uses 2 model evaluations per step (2 * n_steps total NFE).

    Args:
        model: trained UNet
        n_samples: number of images to generate
        n_steps: number of Heun steps
    Returns:
        (n_samples, 1, 28, 28) generated images
    """
    model.eval()
    x = torch.randn(n_samples, 1, 28, 28, device=device)
    dt = 1.0 / n_steps

    for i in range(n_steps):
        t_cur = i / n_steps
        t_next = (i + 1) / n_steps

        t1 = torch.full((n_samples,), t_cur, device=device)
        t2 = torch.full((n_samples,), t_next, device=device)

        # predictor: Euler step
        v1 = model(x, t1)
        x_pred = x + v1 * dt

        # corrector: evaluate at predicted point, average velocities
        v2 = model(x_pred, t2)
        x = x + (v1 + v2) * dt / 2

    return x

### Compare Euler vs. Heun at the Same NFE Budget

Euler with $N$ steps uses $N$ model evaluations. Heun with $N$ steps uses $2N$. For a fair comparison, we compare Euler at $2N$ steps against Heun at $N$ steps (same total NFE).

In [ ]:
nfe_budgets = [4, 10, 20, 50, 100]
n_show = 8

torch.manual_seed(42)
z = torch.randn(n_show, 1, 28, 28, device=device)

fig, axes = plt.subplots(len(nfe_budgets), 2 * n_show + 1,
                         figsize=(2 * n_show * 1.2 + 1, 1.5 * len(nfe_budgets)))
for row in range(len(nfe_budgets)):
    axes[row, n_show].axis('off')

model.eval()
for row, nfe in enumerate(nfe_budgets):
    dt_euler = 1.0 / nfe
    heun_steps = nfe // 2

    # Euler with `nfe` steps
    x_euler = z.clone()
    with torch.no_grad():
        for i in range(nfe):
            t = torch.full((n_show,), i / nfe, device=device)
            x_euler = x_euler + model(x_euler, t) * dt_euler

    # Heun with `nfe // 2` steps (= same NFE)
    x_heun = z.clone()
    if heun_steps > 0:
        x_heun = heun_sample.__wrapped__(model, n_show, n_steps=heun_steps, device=device) if hasattr(heun_sample, '__wrapped__') else z.clone()
        # Manual heun since we need same z
        x_heun = z.clone()
        dt_h = 1.0 / heun_steps
        with torch.no_grad():
            for i in range(heun_steps):
                t1 = torch.full((n_show,), i / heun_steps, device=device)
                t2 = torch.full((n_show,), (i+1) / heun_steps, device=device)
                v1 = model(x_heun, t1)
                x_pred = x_heun + v1 * dt_h
                v2 = model(x_pred, t2)
                x_heun = x_heun + (v1 + v2) * dt_h / 2

    for col in range(n_show):
        axes[row, col].imshow(x_euler[col, 0].cpu().numpy(), cmap='gray')
        axes[row, col].axis('off')
        axes[row, n_show + 1 + col].imshow(x_heun[col, 0].cpu().numpy(), cmap='gray')
        axes[row, n_show + 1 + col].axis('off')
    axes[row, 0].set_ylabel(f'{nfe} NFE', fontsize=10, rotation=0, labelpad=40)

axes[0, n_show // 2].set_title('Euler', fontsize=12, fontweight='bold')
axes[0, n_show + 1 + n_show // 2].set_title('Heun', fontsize=12, fontweight='bold')
plt.suptitle('Euler vs Heun at equal NFE budget', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

\
## 4.2 Noise-Space Interpolation

Since our flow model maps noise $x_0 \sim \mathcal{N}(0, I)$ to images $x_1$, we can **interpolate in noise space** and generate the corresponding images. This produces smooth transitions between generated images.

**Linear interpolation (lerp):**
$$z_\alpha = (1 - \alpha)\, z_1 + \alpha\, z_2$$

**Spherical linear interpolation (slerp)** — better for high-dimensional Gaussians because it stays on the unit hypersphere (preserving the norm of $z$):
$$z_\alpha = \frac{\sin((1-\alpha)\theta)}{\sin\theta}\, z_1 + \frac{\sin(\alpha\theta)}{\sin\theta}\, z_2, \qquad \theta = \arccos\left(\frac{z_1 \cdot z_2}{\|z_1\|\|z_2\|}\right)$$

In [ ]:
def slerp(z1, z2, alpha):
    """
    Spherical linear interpolation between two noise vectors.

    Args:
        z1, z2: (1, C, H, W) noise tensors
        alpha: interpolation weight in [0, 1]
    Returns:
        interpolated noise tensor
    """
    z1_flat = z1.flatten()
    z2_flat = z2.flatten()

    # cosine of angle between vectors
    cos_theta = (z1_flat @ z2_flat) / (z1_flat.norm() * z2_flat.norm())
    cos_theta = cos_theta.clamp(-1, 1)
    theta = torch.acos(cos_theta)

    # handle near-parallel vectors (fall back to lerp)
    if theta.abs() < 1e-6:
        return (1 - alpha) * z1 + alpha * z2

    sin_theta = torch.sin(theta)
    w1 = torch.sin((1 - alpha) * theta) / sin_theta
    w2 = torch.sin(alpha * theta) / sin_theta
    return w1 * z1 + w2 * z2

In [ ]:
# Generate interpolation grids
n_interp = 9
alphas = torch.linspace(0, 1, n_interp)

torch.manual_seed(0)
z1 = torch.randn(1, 1, 28, 28, device=device)
z2 = torch.randn(1, 1, 28, 28, device=device)

fig, axes = plt.subplots(2, n_interp, figsize=(1.5 * n_interp, 3.5))
model.eval()

for col, alpha in enumerate(alphas):
    # Slerp
    z_s = slerp(z1, z2, alpha.item())
    x_s = z_s.clone()
    with torch.no_grad():
        for i in range(50):
            t = torch.full((1,), i / 50, device=device)
            x_s = x_s + model(x_s, t) * (1/50)

    # Lerp
    z_l = (1 - alpha) * z1 + alpha * z2
    x_l = z_l.clone()
    with torch.no_grad():
        for i in range(50):
            t = torch.full((1,), i / 50, device=device)
            x_l = x_l + model(x_l, t) * (1/50)

    axes[0, col].imshow(x_s[0, 0].cpu().numpy(), cmap='gray')
    axes[0, col].axis('off')
    axes[1, col].imshow(x_l[0, 0].cpu().numpy(), cmap='gray')
    axes[1, col].axis('off')
    if col == 0:
        axes[0, col].set_title(f'α=0', fontsize=9)
        axes[1, col].set_title(f'α=0', fontsize=9)
    elif col == n_interp - 1:
        axes[0, col].set_title(f'α=1', fontsize=9)

axes[0, 0].set_ylabel('slerp', fontsize=11, rotation=0, labelpad=35)
axes[1, 0].set_ylabel('lerp', fontsize=11, rotation=0, labelpad=35)
plt.suptitle('Noise-space interpolation: slerp vs lerp', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

\
## 4.3 SDEdit-Style Image Editing

**SDEdit** (Meng et al., 2022) is a simple but powerful idea: to *edit* an image, add noise to it (moving it backward along the flow), then run the flow forward to "denoise" it. The model projects the noisy input onto the learned data manifold.

Given a real image $x_1$:
1. Pick a noise level $t_\text{start} \in (0, 1)$ — controls the edit strength
2. Create noisy version: $x_{t_\text{start}} = (1 - t_\text{start})\, \epsilon + t_\text{start}\, x_1$ where $\epsilon \sim \mathcal{N}(0, I)$
3. Run the ODE forward from $t_\text{start}$ to $1$
4. The output is an edited/cleaned version of $x_1$

- Large $t_\text{start}$ (e.g., 0.9): small edit, stay close to original
- Small $t_\text{start}$ (e.g., 0.3): large edit, model has more freedom to change the image

In [ ]:
@torch.no_grad()
def sdedit(model, x1, t_start, n_steps=50, device='cpu'):
    """
    SDEdit: add noise to an image and denoise via flow.

    Args:
        model: trained UNet
        x1: (B, 1, 28, 28) original images
        t_start: noise level to start from (0 = pure noise, 1 = no edit)
        n_steps: total Euler steps for the full [0, 1] interval
    Returns:
        (B, 1, 28, 28) edited images
    """
    model.eval()
    B = x1.shape[0]
    eps = torch.randn_like(x1)

    # create noisy image at t_start
    x = (1 - t_start) * eps + t_start * x1

    # integrate from t_start to 1
    dt = 1.0 / n_steps
    # find the step index corresponding to t_start
    start_step = int(t_start * n_steps)

    for i in range(start_step, n_steps):
        t = torch.full((B,), i / n_steps, device=device)
        v = model(x, t)
        x = x + v * dt

    return x

In [ ]:
# Demonstrate SDEdit at different noise levels
test_images, _ = next(iter(test_loader))
x1 = test_images[:8].to(device)
t_starts = [0.9, 0.7, 0.5, 0.3, 0.1]

fig, axes = plt.subplots(len(t_starts) + 1, 8, figsize=(12, 1.8 * (len(t_starts) + 1)))

# Show originals
for col in range(8):
    axes[0, col].imshow(x1[col, 0].cpu().numpy(), cmap='gray')
    axes[0, col].axis('off')
axes[0, 0].set_ylabel('original', fontsize=10, rotation=0, labelpad=50)

# Show SDEdit at each t_start
for row, t_s in enumerate(t_starts):
    edited = sdedit(model, x1, t_start=t_s, n_steps=50, device=device)
    for col in range(8):
        axes[row + 1, col].imshow(edited[col, 0].cpu().numpy(), cmap='gray')
        axes[row + 1, col].axis('off')
    axes[row + 1, 0].set_ylabel(f't={t_s}', fontsize=10, rotation=0, labelpad=35)

plt.suptitle('SDEdit: varying noise level (t close to 1 = small edit)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

---
# Part 5: Pretrained Flow Matching Models

So far we trained small models on MNIST. Here we load **pretrained** flow matching models on CIFAR-10 and ImageNet to see what full-scale training achieves — and run sampling experiments without needing to train.

## 5.1 Pretrained CIFAR-10 Flow Matching

We load a flow matching model trained on CIFAR-10 (32×32, color) via Hugging Face `diffusers`. This model uses a `UNet2DModel` with `FlowMatchEulerDiscreteScheduler`.

In [ ]:
!pip install -q diffusers accelerate

In [ ]:
from diffusers import DiffusionPipeline
import numpy as np

# Load pretrained CIFAR-10 flow matching model
cifar_pipe = DiffusionPipeline.from_pretrained("FrankCCCCC/cfm-cifar10-32")
cifar_pipe = cifar_pipe.to(device)

print(f"Model type: {type(cifar_pipe.unet).__name__}")
print(f"Scheduler:  {type(cifar_pipe.scheduler).__name__}")
print(f"UNet params: {sum(p.numel() for p in cifar_pipe.unet.parameters()):,}")

In [ ]:
# Generate CIFAR-10 samples with the default pipeline
output = cifar_pipe(batch_size=32, num_inference_steps=50)
images = output.images  # list of PIL images

# Display
fig, axes = plt.subplots(4, 8, figsize=(14, 7))
for i, ax in enumerate(axes.flat):
    ax.imshow(np.array(images[i]))
    ax.axis('off')
plt.suptitle('Pretrained CIFAR-10 Flow Matching (50 steps)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 5.2 Sampling Steps vs. Quality on CIFAR-10

With the pretrained model, let's see how sample quality degrades as we reduce the number of Euler steps.

In [ ]:
step_counts_cifar = [1, 2, 5, 10, 25, 50, 100]
n_show_cifar = 8

fig, axes = plt.subplots(len(step_counts_cifar), n_show_cifar,
                         figsize=(1.5 * n_show_cifar, 1.5 * len(step_counts_cifar)))

# Use a fixed seed for fair comparison
for row, n_steps in enumerate(step_counts_cifar):
    generator = torch.Generator(device=device).manual_seed(42)
    output = cifar_pipe(
        batch_size=n_show_cifar,
        num_inference_steps=n_steps,
        generator=generator,
    )
    for col in range(n_show_cifar):
        axes[row, col].imshow(np.array(output.images[col]))
        axes[row, col].axis('off')
    axes[row, 0].set_ylabel(f'{n_steps}', fontsize=10, rotation=0, labelpad=25)

fig.text(0.02, 0.5, 'Steps', fontsize=12, rotation=90, va='center')
plt.suptitle('CIFAR-10: Sample quality vs. Euler steps', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

\
## 5.3 Temperature Scaling

By scaling the initial noise $x_0 = \tau \cdot \epsilon$ where $\epsilon \sim \mathcal{N}(0, I)$, we control the **temperature** of generation:

- $\tau < 1$: lower diversity, samples cluster near the mode (sharper but less varied)
- $\tau = 1$: standard generation
- $\tau > 1$: higher diversity, but may reduce quality

In [ ]:
temperatures = [0.5, 0.7, 0.85, 1.0, 1.2, 1.5]
n_show_temp = 8

fig, axes = plt.subplots(len(temperatures), n_show_temp,
                         figsize=(1.5 * n_show_temp, 1.5 * len(temperatures)))

cifar_unet = cifar_pipe.unet
cifar_scheduler = cifar_pipe.scheduler

for row, temp in enumerate(temperatures):
    # Manual sampling with temperature scaling
    torch.manual_seed(42)
    x = torch.randn(n_show_temp, 3, 32, 32, device=device) * temp

    cifar_scheduler.set_timesteps(50)
    for t in cifar_scheduler.timesteps:
        t_batch = t.expand(n_show_temp).to(device)
        with torch.no_grad():
            model_output = cifar_unet(x, t_batch).sample
        x = cifar_scheduler.step(model_output, t, x).prev_sample

    # Convert to displayable images
    x_display = (x.clamp(-1, 1) + 1) / 2  # [-1,1] -> [0,1]
    for col in range(n_show_temp):
        axes[row, col].imshow(x_display[col].permute(1, 2, 0).cpu().numpy())
        axes[row, col].axis('off')
    axes[row, 0].set_ylabel(f'τ={temp}', fontsize=10, rotation=0, labelpad=35)

plt.suptitle('CIFAR-10: Temperature scaling (τ)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 5.4 Class-Conditional ImageNet Generation (Demo)

The **Diffusion Transformer (DiT)** generates 256×256 ImageNet images conditioned on any of 1000 classes. While DiT uses a DDPM-style formulation (not flow matching), its architecture — a transformer operating on image patches with adaptive layer norm — is the same as what **Stable Diffusion 3** and **FLUX** use with flow matching. This demonstrates what these architectures can produce at scale.

> **Note:** This requires ~8GB GPU memory (fp16). If you're on a Colab T4 (16GB), this should work.

In [ ]:
from diffusers import DiTPipeline, DPMSolverMultistepScheduler

dit_pipe = DiTPipeline.from_pretrained("facebook/DiT-XL-2-256", torch_dtype=torch.float16)
dit_pipe = dit_pipe.to(device)
# Use a faster scheduler
dit_pipe.scheduler = DPMSolverMultistepScheduler.from_config(dit_pipe.scheduler.config)

print(f"DiT params: {sum(p.numel() for p in dit_pipe.transformer.parameters()):,}")

In [ ]:
# Generate images for various ImageNet classes
class_names = [
    "golden retriever", "red fox", "tabby cat", "monarch butterfly",
    "jellyfish", "volcano", "lighthouse", "acoustic guitar",
]
class_ids = dit_pipe.get_label_ids(class_names)

images = dit_pipe(
    class_labels=class_ids,
    num_inference_steps=25,
    guidance_scale=4.0,
).images

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, (img, name) in enumerate(zip(images, class_names)):
    ax = axes[i // 4, i % 4]
    ax.imshow(np.array(img))
    ax.set_title(name, fontsize=11)
    ax.axis('off')
plt.suptitle('DiT-XL/2: Class-conditional ImageNet 256×256', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

### Guidance Scale Sweep on ImageNet

Same idea as Part 2 (CFG on MNIST), but now on ImageNet at 256×256 resolution.

In [ ]:
# Guidance scale sweep for a single class
guidance_scales = [1.0, 2.0, 4.0, 7.5, 15.0]
target_class = "golden retriever"
class_id = dit_pipe.get_label_ids([target_class])
n_per_scale = 4

fig, axes = plt.subplots(len(guidance_scales), n_per_scale,
                         figsize=(3.5 * n_per_scale, 3.5 * len(guidance_scales)))

for row, w in enumerate(guidance_scales):
    images = dit_pipe(
        class_labels=class_id * n_per_scale,
        num_inference_steps=25,
        guidance_scale=w,
        generator=torch.Generator(device=device).manual_seed(42),
    ).images
    for col in range(n_per_scale):
        axes[row, col].imshow(np.array(images[col]))
        axes[row, col].axis('off')
    axes[row, 0].set_ylabel(f'w={w}', fontsize=11, rotation=0, labelpad=40)

plt.suptitle(f'Guidance scale sweep: "{target_class}"', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

---
# Summary

In this notebook we covered:

1. **Unconditional OT-CFM on images** — the same linear interpolation and velocity matching from the Gaussian walkthrough, but with a U-Net architecture for 2D images.

2. **Class-conditional generation with CFG** — by adding a class embedding and training with random label dropout, we can control which digit to generate. The guidance scale $w$ trades off diversity for class fidelity.

3. **Rectified Flow (Reflow)** — by re-pairing noise and images using the trained model, we can retrain on "straightened" couplings, producing a model that generates better samples with fewer Euler steps.

4. **Advanced sampling** — Heun's method gives better quality at the same NFE budget; slerp interpolation produces smooth transitions; SDEdit enables image editing via noise injection.

5. **Pretrained models** — flow matching on CIFAR-10 and class-conditional ImageNet generation with DiT, demonstrating what these methods achieve at scale.

## Open Problems for the Hackathon

**Sampling & Solvers:**
- Implement **RK4** (4th-order Runge-Kutta) and compare against Euler/Heun
- Try **adaptive** step-size solvers (`torchdiffeq.odeint` with `dopri5`) — let the solver pick the step size
- Experiment with **stochastic samplers**: add noise during the ODE integration for diversity

**Reflow & Distillation:**
- Run reflow **multiple times** (2-Rectified Flow, 3-Rectified Flow) — does straightness keep improving?
- Implement **flow distillation**: train a student to match the teacher's output in 1-2 steps
- Compare trajectory straightness across the 1-RF, 2-RF, 3-RF checkpoints from [gnobitab/RectifiedFlow](https://github.com/gnobitab/RectifiedFlow)

**Conditioning & Guidance:**
- Try a **guidance schedule**: vary $w$ across timesteps (e.g., high guidance early, lower late)
- Implement **classifier guidance** (train a separate classifier on noisy images)
- Explore **CFG-Zero\*** (Deng et al., 2025): improved CFG for flow matching models

**Editing & Inverse Problems:**
- **Inpainting**: mask a region, condition the flow on the unmasked pixels
- **Super-resolution**: flow from low-res to high-res images
- **RF-Inversion** (Rout et al.): invert a real image to its noise vector, then edit and re-generate
- **Posterior sampling** with `deepinv`: use flow matching for inverse problems (deblurring, inpainting, compressed sensing)

**Architecture & Scale:**
- Replace the U-Net with a **Diffusion Transformer (DiT)** and compare
- Try **latent flow matching**: encode with a VAE, do flow matching in latent space (as in Stable Diffusion 3)
- Scale to **CIFAR-10** from scratch: increase model size, train longer, measure FID

**Theory:**
- Compute **exact log-likelihoods** via the continuous normalizing flow formulation (trace of Jacobian)
- Compare different **probability paths**: OT vs. diffusion-style vs. cosine schedules
- Implement **Riemannian flow matching** on non-Euclidean spaces (e.g., sphere, SO(3))